# 🧬 DeepDTA+ CNN Training Notebook (100 Epochs, T4 GPU)

**Team G1542** — SaiCharan, Havisha, Abhinov, Rishitha, Hariharan

This notebook trains the faithful PyTorch DeepDTA CNN (Öztürk et al., 2018) for **100 full epochs** on a **T4 GPU** using the benchmark Davis dataset.

## Setup
1. In Google Colab, go to **Runtime → Change runtime type → T4 GPU**.
2. Run all cells in sequence.
3. Download the generated checkpoint weights (`deepdta_cnn_best.pt`) and place into `results/checkpoints/`.

In [ ]:
# Step 1: Clone the DeepDTA-Plus repository
!git clone https://github.com/SC-Charan/DeepDTA-Plus.git
%cd DeepDTA-Plus

In [ ]:
# Step 2: Install dependencies & verify GPU
!pip install torch numpy scikit-learn -q

import sys
sys.path.insert(0, 'src')

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU device: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('WARNING: Running on CPU. Go to Runtime -> Change runtime type -> T4 GPU.')

In [ ]:
# Step 3: Load the Davis dataset
from dtaplus.data import DavisData

print('Loading Davis dataset...')
data = DavisData(data_dir='data/davis')
print(f'  Unique Drugs: {data.num_drugs}')
print(f'  Unique Targets: {data.num_proteins}')
print(f'  Affinity Pairs: {data.num_pairs}')
print(f'  Drug tensor shape: {data.encoded_drugs.shape}')
print(f'  Protein tensor shape: {data.encoded_proteins.shape}')

In [ ]:
# Step 4: Run 100-Epoch Training on GPU
from dtaplus.train import train_cnn
import json

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Launching 100-epoch CNN training on {device}...')

results = train_cnn(
    data=data,
    val_fold_idx=0,
    epochs=100,
    lr=0.001,
    batch_size=256,
    device=device,
    smoke=False, # Full training
    seed=42,
    save_dir='results/checkpoints',
)

print('\n' + '=' * 50)
print('TRAINING SUMMARY:')
print(json.dumps(results, indent=2))
print('=' * 50)

In [ ]:
# Step 5: Verify Checkpoint & Test Inference
import os
from dtaplus.models.cnn import DeepDTACNN
from dtaplus.data import encode_smiles, encode_protein

ckpt_path = 'results/checkpoints/deepdta_cnn_best.pt'
assert os.path.exists(ckpt_path), 'Checkpoint file was not found!'
print(f'Checkpoint saved successfully: {ckpt_path} ({os.path.getsize(ckpt_path)/(1024*1024):.2f} MB)')

# Quick inference with Dasatinib on ABL1
model = DeepDTACNN()
ckpt = torch.load(ckpt_path, map_location='cpu')
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

dasatinib_smi = 'CC1=C(C(=CC=C1)Cl)NC(=O)C2=CN=C(S2)NC3=CC(=NC(=N3)C)N4CCN(CC4)CCO'
abl1_seq = data.protein_sequences[0]
s_tensor = torch.from_numpy(encode_smiles(dasatinib_smi)).unsqueeze(0).long()
p_tensor = torch.from_numpy(encode_protein(abl1_seq)).unsqueeze(0).long()

with torch.no_grad():
    pred = model(s_tensor, p_tensor).item()
print(f'Sample Inference - Dasatinib + ABL1 pKd: {pred:.3f} (Kd: {10**(9-pred):.2f} nM)')

In [ ]:
# Step 6: Download the trained weights to your local machine
from google.colab import files

files.download('results/checkpoints/deepdta_cnn_best.pt')
files.download('results/checkpoints/training_summary.json')
print('Checkpoints ready for download!')